# SPX probability signal data-quality audit

This notebook verifies which probability and execution labels can be used by the first production shadow. Oracle is the factual authority; no broker credentials or account data are loaded.

In [1]:
from pathlib import Path
import json

artifact_path = Path('../probability-signal-data-quality-2026-08-05.json')
audit = json.loads(artifact_path.read_text(encoding='utf-8'))
assert audit['schema_version'] == 'probability_signal_data_quality.v1'
assert audit['authority'] == 'oracle:129.146.3.211'
audit['code_revision']

'08485f4a80b520d17bde382745c4c039832c1468'

In [2]:
physical = audit['physical_path_outcomes']
reach = audit['displayed_quote_reach']
broker = audit['broker_order_lifecycle']
pnl = audit['net_pnl']

eligibility = {
    'physical_followthrough': physical['distinct_events'] >= 30 and physical['trading_days'] >= 5,
    'displayed_quote_reach_calibration': (reach['clean_reached'] + reach['clean_not_reached']) >= 30 and reach['runtime_lake_conflicts'] == 0,
    'actual_fill_model': broker['orders_at_risk'] >= 30 and broker['full_or_partial_fills'] > 0,
    'net_pnl_quantiles': pnl['strict_current_policy_training_samples'] >= 30,
}
eligibility

{'physical_followthrough': True,
 'displayed_quote_reach_calibration': False,
 'actual_fill_model': False,
 'net_pnl_quantiles': False}

In [3]:
assert eligibility == {
    'physical_followthrough': True,
    'displayed_quote_reach_calibration': False,
    'actual_fill_model': False,
    'net_pnl_quantiles': False,
}
assert audit['research_history']['research_context_append_only'] is False
assert audit['decision']['automatic_ordering'] is False

label_contract = {
    'P': 'prior-trading-day directional terminal return at 300 seconds',
    'Q': 'same-event short-horizon ATM N(d2) risk-neutral proxy',
    'reach': 'displayed_quote_reach_proxy; never actual fill',
    'net_pnl': None,
    'selection': 'NoTrade until exact-cost labels exist',
}
label_contract

{'P': 'prior-trading-day directional terminal return at 300 seconds',
 'Q': 'same-event short-horizon ATM N(d2) risk-neutral proxy',
 'reach': 'displayed_quote_reach_proxy; never actual fill',
 'net_pnl': None,
 'selection': 'NoTrade until exact-cost labels exist'}

## Frozen first-production conclusion

Publish an uncalibrated physical follow-through baseline with an explicit uncertainty interval. Persist every decision-time forecast append-only. Quote reach remains a small, conflicted displayed-liquidity proxy; actual fill and net-PnL quantiles stay unavailable. NoTrade is a valid model selection, and automatic ordering remains disabled.